# Week 6 — Mini trial questions (CourtListener)

**Name:** Ranjitha Rangaswamy  
**Dataset:** CourtListener export (`courtlistener_week5_repull_40k.csv`)  
**Date:** May 2026

This notebook is **Week 6 only**: simple trial questions you could ask a court-data pull (counts and slices with pandas).

**Not in this notebook:** Mini Project 1 questions about judges, citation frequency by court level, top cited judgments in a jurisdiction, or dataset-wide authority titles. Those live in **`MiniProject1/Miniproject1.ipynb`**.

Work through the cells in order. Write a short interpretation under each answer.


In [1]:
# Setup — run first
import subprocess, sys
from pathlib import Path

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.graph_objects as go

possible_paths = [
    Path("courtlistener_week5_repull_40k.csv"),
    Path("../MiniProject1/data/courtlistener.csv"),
    Path("../../MiniProject1/data/courtlistener.csv"),
]
csv_path = next((p for p in possible_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find courtlistener CSV in Week6_files or MiniProject1/data/")

df = pd.read_csv(csv_path)
df["date_of_decision"] = pd.to_datetime(df["date_of_decision"], errors="coerce")
print(f"Loaded: {csv_path.resolve()}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


Loaded: /Users/rnr6/Documents/HCDE/hcde530/week 6/Week6_files/courtlistener_week5_repull_40k.csv
Shape: 39,040 rows × 7 columns


,dataset,case,judge,court,court_id,date_of_decision,cite_count
0,King County (Wash. Ct. App.),"State Of Washington, V. Justin R. Smith",NaN,Court of Appeals of Washington,washctapp,2025-07-21,0
1,King County (Wash. Ct. App.),"Chen Wang And Liyin Xue, V. Dongmei Huang",NaN,Court of Appeals of Washington,washctapp,2025-08-25,0
2,King County (Wash. Ct. App.),In the Matter of the Detention of C.E.,NaN,Court of Appeals of Washington,washctapp,2025-09-09,0
3,King County (Wash. Ct. App.),"Erwin Chappel, Respondent/cr-appellants V. Dou...",NaN,Court of Appeals of Washington,washctapp,2025-09-15,0
4,King County (Wash. Ct. App.),"Alaska Airlines, V. Hillary Spanjer",NaN,Court of Appeals of Washington,washctapp,2025-09-15,0


---

## Trial question 1 — How many opinion rows were decided in 2025?

*Plain ask:* “How many cases in Seattle in 2025?”  

This CSV is **CourtListener opinions**, not every filing in a docket. Count rows where `date_of_decision` is in **2025** and show which **court_id** they belong to.


In [2]:
in_2025 = df["date_of_decision"].dt.year == 2025
n_2025 = int(in_2025.sum())
print(f"Rows with decision year 2025: {n_2025:,} ({100 * n_2025 / len(df):.1f}% of file)")

by_court_2025 = (
    df.loc[in_2025]
    .groupby("court_id", dropna=False)
    .size()
    .sort_values(ascending=False)
)
print("\n2025 rows by court_id:")
print(by_court_2025)


Rows with decision year 2025: 36,000 (92.2% of file)

2025 rows by court_id:
court_id
washctapp    36000
dtype: int64


**Interpretation:** In this pull, 2025 is almost entirely state appellate (`washctapp`, King County search). Federal W.D. Wash. rows in this file are mostly 2018–2019, not 2025.


---

## Trial question 2 — In 2025, how many rows look family-related by case title?

*Plain ask:* “How many family law cases in 2025 in Seattle?”  

There is **no** family-law column. Use a **keyword proxy** on `case` (guardianship, dissolution, domestic, parentage, marriage, spousal, and similar). Report **row count** and **how many unique titles** matched (duplicate rows per caption are common).


In [3]:
FAMILY_TITLE_PATTERN = (
    r"guardianship|dissolution|marital|parentage|paternity|domestic|"
    r"marriage|spousal|custody|divorce"
)

df_2025 = df[df["date_of_decision"].dt.year == 2025].copy()
family_mask = df_2025["case"].astype(str).str.contains(
    FAMILY_TITLE_PATTERN, case=False, na=False, regex=True
)
n_family_rows = int(family_mask.sum())
n_family_titles = df_2025.loc[family_mask, "case"].nunique()

print(f"2025 rows total: {len(df_2025):,}")
print(f"2025 rows with family-related title keywords: {n_family_rows:,}")
print(f"Unique matching case titles: {n_family_titles:,}")

if n_family_titles:
    print("\nTop matching titles (by row count):")
    print(
        df_2025.loc[family_mask, "case"]
        .value_counts()
        .head(10)
        .to_string()
    )
else:
    print("\nNo titles matched the keyword list in 2025.")


2025 rows total: 36,000
2025 rows with family-related title keywords: 1,800
Unique matching case titles: 1

Top matching titles (by row count):
case
Guardianship Of J.S.    1800


**Interpretation:** A large row count can be **one caption repeated** many times. Say both row count and unique titles when you report “how many family cases.” This is not the same as a court’s official case-type code.


---

## Trial question 3 — How many federal (W.D. Wash.) opinions in 2025?

*Plain ask:* “How many federal court cases in Seattle in 2025?”  

Filter **2025** and `court_id == "wawd"` (U.S. District Court, Western District of Washington).


In [4]:
federal_2025 = df[
    (df["date_of_decision"].dt.year == 2025) & (df["court_id"] == "wawd")
]
n_federal_2025 = len(federal_2025)
print(f"Federal (wawd) rows with decision year 2025: {n_federal_2025:,}")

if n_federal_2025 == 0:
    federal_any_year = df[df["court_id"] == "wawd"]
    years = (
        federal_any_year["date_of_decision"]
        .dt.year.dropna()
        .astype(int)
        .value_counts()
        .sort_index()
    )
    print("\nFederal rows in this file by decision year (for context):")
    print(years.to_string())


Federal (wawd) rows with decision year 2025: 0

Federal rows in this file by decision year (for context):
date_of_decision
2018     152
2019    2888


**Interpretation:** Zero federal 2025 rows means this **export window** does not cover that question, not that Seattle federal court had no activity in 2025. Check which years `wawd` actually has before drawing conclusions.


---

## Trial question 4 — In 2025, how many criminal-style appeals vs other captions?

*Plain ask:* “How much of the 2025 docket is criminal vs civil?”  

Use a **title pattern**: rows whose `case` looks like **State of Washington v. [defendant]** count as criminal-style appeals in this proxy. Everything else in 2025 is labeled “other” here (not a perfect civil/family split).


In [5]:
CRIMINAL_TITLE_PATTERN = r"State [Oo]f Washington,\s*V\.|State of Washington v\."

criminal_mask = df_2025["case"].astype(str).str.contains(
    CRIMINAL_TITLE_PATTERN, case=False, na=False, regex=True
)
n_criminal = int(criminal_mask.sum())
n_other = len(df_2025) - n_criminal

print(f"2025 rows (criminal-style title): {n_criminal:,}")
print(f"2025 rows (other captions): {n_other:,}")
print(
    f"Share criminal-style: {100 * n_criminal / len(df_2025):.1f}%"
    if len(df_2025)
    else "No 2025 rows"
)

print("\nSample non-criminal 2025 titles:")
sample_other = (
    df_2025.loc[~criminal_mask, "case"].drop_duplicates().head(8).tolist()
)
for title in sample_other:
    print(f"  - {title}")


2025 rows (criminal-style title): 12,600
2025 rows (other captions): 23,400
Share criminal-style: 35.0%

Sample non-criminal 2025 titles:
  - Chen Wang And Liyin Xue, V. Dongmei Huang
  - In the Matter of the Detention of C.E.
  - Erwin Chappel, Respondent/cr-appellants V. Douglas Johnson, Appellant/cr-respondents
  - Alaska Airlines, V. Hillary Spanjer
  - Samantha Snodderly, V. Bradley Shockey
  - Clifton A. Little Ii Et Ano, V. Hardie-tynes Co. Inc.
  - Guardianship Of J.S.
  - Lisa Earl, V. City Of Tacoma, Scott Campbell


**Interpretation:** “State v.” is a rough criminal proxy only. Family-related rows from question 2 can still sit in the “other” bucket (for example guardianship captions). For stakeholders, name the proxy you used.


---

## Where Mini Project 1 lives (different questions)

| Item | Location |
|------|----------|
| MP1: judges, citations by court level, top cited cases, authority titles | `MiniProject1/Miniproject1.ipynb` |
| MP1 static chart JPGs | `MiniProject1/images/` |
| MP1 Dash app (not Week 6 trial questions) | `week 6/courtlistener_mp1a_dash.py` |


---

## Week 6 charts — interactive 3D + static export

Three **interactive 3D** Plotly charts (rotate, zoom, hover). Each answers one trial question above. Static **PNG** and interactive **HTML** are written to this folder (`Week6_files/`).

Run the cell below after the trial-question cells (or anytime after setup).

In [6]:
# 3D charts — display in notebook and export PNG + HTML
import sys

week6_dir = Path.cwd().resolve()
if not (week6_dir / "week6_charts.py").is_file():
    week6_dir = week6_dir.parent
if str(week6_dir) not in sys.path:
    sys.path.insert(0, str(week6_dir))

import week6_charts as w6

chart_df = w6.load_df()
fig1 = w6.chart1_opinions_by_year_court_3d(chart_df)
fig2 = w6.chart2_2025_criminal_vs_other_3d(chart_df)
fig3 = w6.chart3_2025_family_keyword_3d(chart_df)

for label, fig in [
    ("Chart 1 — year × court (trial Q1)", fig1),
    ("Chart 2 — criminal vs other 2025 (trial Q4)", fig2),
    ("Chart 3 — family keywords 2025 (trial Q2)", fig3),
]:
    print(label)
    fig.show()

w6.export_all()
print(f"\nExports saved under: {w6.OUT_DIR.resolve()}")

Chart 1 — year × court (trial Q1)


Chart 2 — criminal vs other 2025 (trial Q4)


Chart 3 — family keywords 2025 (trial Q2)


Wrote /Users/rnr6/Documents/HCDE/hcde530/week 6/Week6_files/week6_chart1_opinions_by_decision_year.png
Wrote /Users/rnr6/Documents/HCDE/hcde530/week 6/Week6_files/week6_chart1_opinions_by_decision_year.html
Wrote /Users/rnr6/Documents/HCDE/hcde530/week 6/Week6_files/week6_chart2_2025_criminal_vs_other_captions.png
Wrote /Users/rnr6/Documents/HCDE/hcde530/week 6/Week6_files/week6_chart2_2025_criminal_vs_other_captions.html
Wrote /Users/rnr6/Documents/HCDE/hcde530/week 6/Week6_files/week6_chart3_2025_family_title_keyword_rows.png
Wrote /Users/rnr6/Documents/HCDE/hcde530/week 6/Week6_files/week6_chart3_2025_family_title_keyword_rows.html

Exports saved under: /Users/rnr6/Documents/HCDE/hcde530/week 6/Week6_files
